# Step 19.4 — Apple Silicon resource control：先估算，再明确观察口径

在仓库根目录同步母版：

```bash
git pull
```

再创建独立工作副本；时间戳与 `cp -n` 避免覆盖笔记：

```bash
cp -n notebooks/85_step19_4_apple_silicon_resource_control.ipynb notebooks/work/85_step19_4_apple_silicon_resource_control_$(date +%Y%m%d_%H%M%S)_work.ipynb
```

以上是未来使用的命令参考，本次未执行。环境沿用项目 Python 3.12 / `.venv` 与 **MLX v0.32.2**。全部代码单元未执行、输出为空；没有读取语料、构造模型、训练、测量或修改系统设置。

2026-11-17 教学修订 **1/2**：去掉同一单元内重复的模型定义，将资源概念和小 helper 放在正文，把一套完整旧依赖放在明确的可选附录。数据读取改为需主动调用的函数。旧初稿为0/2，其他小节计数不变。本轮完成19.4后停止。


## 阅读路线：这一节到底新增什么？

[19.1](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/50aed835a74af0fe380b39b854329994908630c1/notebooks/82_step19_1_hyperparameter_baseline.ipynb)给了候选配置，[19.2](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/50aed835a74af0fe380b39b854329994908630c1/notebooks/83_step19_2_train_validation_loop.ipynb)区分训练与验证，[19.3](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/50aed835a74af0fe380b39b854329994908630c1/notebooks/84_step19_3_training_stability.ipynb)在更新前检查有限值和梯度。本节增加一个不同的问题：**这份候选会产生怎样的工作量，将来应从哪里观察资源压力？** 数值有限不证明资源足够，短步骤结束也不证明持续训练稳定。

| 区域 | 要做的阅读 | 需要执行模型吗？ |
|---|---|---|
| 正文1–6节：本节重点 | 区分统计口径，手算一张score张量，理解候选修改与观察顺序 | 不需要；小函数只定义，不自动调用 |
| 附录A–C：已学前置／可选完整代码 | 以后新kernel确需恢复完整模型时，按依赖顺序使用 | 只是导入、常量与定义；不自动读数据或创建模型 |
| 附录中的注释调用例 | 以后自己明确决定读取缓存或初始化对象时才使用 | 取消注释或主动调用才发生相应动作；没有训练循环入口 |

理解本节不必重读完整MiniGPT。附录保留它是为了未来工作副本自包含；它不是19.4的新架构，也不是要求现在把数百行代码直接运行。训练和验证循环仍按前节各自的工作副本处理，不混进本节正文。


## 1. Unified memory：先分清三种不同的量

Apple Silicon 的 **unified memory（统一内存）** 允许CPU和GPU访问同一内存池。它避免把模型想成两份相互独立的“CPU内存+专用显存”，但共享不等于容量无限，也不意味着所有资源能由同一个API统计。

| 量 | 回答什么问题 | 不能推出什么 |
|---|---|---|
| **Logical payload（逻辑数据量）** | 给定shape和dtype，这些元素按值需要多少字节？ | 不等于实际进程峰值 |
| **MLX allocator counters（分配器计数）** | 当前进程的MLX buffer在使用、待复用与历史高点分别多少？ | 不包含整机所有Python对象、其他程序与系统资源 |
| **System memory pressure / Swap** | macOS层面的内存压力与磁盘交换使用 | 不能仅从某个MLX数组的字节数反推 |

**Swap（交换空间）** 是操作系统使用磁盘支持内存管理的空间，不是MLX保留的空闲buffer。Activity Monitor的Memory Pressure、Memory Used、Swap Used应单独观察；这些指标和MLX的active/cache/peak不能混成一个“总显存”列。

当前只是解释口径，没有任何系统读数。[固定版本统一内存说明](https://github.com/ml-explore/mlx/blob/v0.32.2/docs/src/usage/unified_memory.rst)与[Apple Activity Monitor说明](https://support.apple.com/guide/activity-monitor/view-memory-usage-actmntr1004/mac)分别对应这两个层面。


## 2. 只手算一张张量：attention scores有多大？

沿用19.1候选：batch大小 **B=4**，query头数 **H=4**，每条窗口长度 **T=64**，模型宽度 **C=64**、层数 **N=2**。B决定同时处理几条窗口；T决定一条窗口有几个位置；当前每头维度是C/H=16。

原MiniGPT显式计算每个query与每个key的分数，因此一层的score shape为 **(B,H,T,T)**。两条T轴分别是query位置和key位置，不是多写了一次同一个轴。假定这些score以float32表示，一项为4 bytes：

$$
N_{score}=BHT^2=4\times4\times64\times64=65,536,
$$
$$
S_{score}=4N_{score}=262,144\;\text{bytes}
=0.25\;\text{MiB},\qquad 1\;\text{MiB}=2^{20}\;\text{bytes}.
$$

这0.25 MiB仅是**一张score张量**的逻辑payload。它不是本模型的峰值：还可能有softmax结果、Q/K/V、激活、参数、梯度、Adam状态及编译/缓存等；它们的生命周期也未必相同，不能把每个源码中出现的数组简单全加起来当峰值。

若有P个float32参数，单份参数为4P bytes；同shape的Adam一阶/二阶矩合计8P bytes，若同dtype梯度此时也存活则再有4P bytes。这仍只是在明确假设下列出几类payload，不含全部优化器标量、激活与系统开销，不是实测预算。

下面的 `math.prod(shape)` 只把Python整数维度连乘，再乘4。它不创建MLX数组，也不会去读机器内存。输入约定是合法的非负整数shape；本节不把它扩成通用资源检测器。


In [ ]:
import math

baseline_candidate = {'model_width':64,'layers':2,'heads':4,'block_size':64,'batch_size':4,'learning_rate':3e-4}

def logical_float32_bytes(shape):
    """Arithmetic estimate only; does not allocate or measure memory."""
    elements = math.prod(shape)
    return elements * 4


### 把同一个估算用于候选比较

仍只比较上述float32 score，不改变H：

| 候选 | Score shape | 手算逻辑payload | 能说明什么 |
|---|---|---:|---|
| 原候选 | (4,4,64,64) | 0.25 MiB | 本次比较起点 |
| 只把B改为2 | (2,4,64,64) | 0.125 MiB | 这一张量的元素数减半 |
| 只把T改为32 | (4,4,32,32) | 0.0625 MiB | 这一张量的元素数变为四分之一 |

这些是同一公式的纸面值，没有分配数组或运行对比。实际峰值与速度不一定按此比例变化。减小B也改变每步监督位置数；减小T还改变上下文范围，不能把它们描述成完全相同的训练任务。

宽度C和层数N影响参数与激活。改C时必须仍满足C能被H整除；改架构或位置表容量后，要建立匹配的新模型/优化器，不能仅改一个字典就宣称旧对象已缩小。


## 3. Conservative sizing：先改候选字典，不动运行中的对象

`propose_resource_candidate` 返回新字典。`dict(config)` 复制当前这些标量配置，后面的修改不回写原字典；`None` 表示保持原值。`*` 后的参数必须按名字给出，方便看清一次只改了哪个维度。它保留当前heads与learning_rate。

原接口在修改block_size时同时写入一个 `max_seq_len` 键，用于表达相同的候选容量；真正调用模型时，附录的 `make_model_kwargs` 仍从 **block_size** 映射到MiniGPT的 **max_seq_len**。字典里的别名不负责改变已经存在的position embedding。

末尾检查尺寸为正且width可被heads整除。这是对本课整数配置的基本约束，不证明某个候选能在机器上安全训练。未来若尝试改变一个候选，保留原配置和变更项，避免同时改B/T/C/N后无法解释差异。


In [ ]:
def propose_resource_candidate(config, *, batch_size=None, block_size=None, model_width=None, layers=None):
    """Return a new candidate dictionary; does not instantiate a model or change system settings."""
    candidate = dict(config)
    if batch_size is not None: candidate['batch_size'] = batch_size
    if block_size is not None:
        candidate['block_size'] = block_size
        candidate['max_seq_len'] = block_size
    if model_width is not None: candidate['model_width'] = model_width
    if layers is not None: candidate['layers'] = layers
    if candidate['model_width'] % candidate['heads'] != 0:
        raise ValueError('model_width must be divisible by heads')
    if min(candidate['batch_size'], candidate['block_size'], candidate['model_width'], candidate['layers']) < 1:
        raise ValueError('size parameters must be positive')
    return candidate


例如以后主动调用 `propose_resource_candidate(baseline_candidate, batch_size=2)`，得到的是一份B=2的候选，原候选B仍为4。调用本身不创建模型、不采样，也不更改系统资源限制。此处没有调用它；表中的数值来自上面的手算。

选择更小候选只是在降低特定工作量。它不是已测得的安全上限，不保证零Swap、低温、固定速度或收敛。本节也不调整memory/cache/wired limit，不使用系统权限设置来代替观察。


## 4. 观察路径：先完成指定工作，再读取对应计数

[18.5 内存计数](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/50aed835a74af0fe380b39b854329994908630c1/notebooks/81_step18_5_memory_instrumentation.ipynb)已区分下面三个顶层API；固定v0.32.2均无参数、返回Python整数，单位为bytes：

| 返回字段 | 实际API | 范围 |
|---|---|---|
| active_bytes | `mx.get_active_memory()` | 正在使用的MLX分配器buffer |
| cache_bytes | `mx.get_cache_memory()` | 暂不使用、保留以待复用的buffer |
| peak_bytes | `mx.get_peak_memory()` | 进程开始或最近reset以来的active高点 |

同一观察点的active与cache是当前量；peak是历史高点，不能再加进去算总量，也不能把两个累计peak相减当作这一段的峰值。本节不调用reset；若未来需要窗口峰值，沿用18.5包含起点active的完整口径。

下面保留原函数名 `mlX_memory_snapshot_bytes`，避免改动现有调用接口。**定义函数不读取计数，调用函数才读取。** 它只返回三项读数，不清理缓存、不更改限制、不读取温度或Swap。


In [ ]:
import mlx.core as mx

def mlX_memory_snapshot_bytes():
    """MLX allocator counters only; all values are bytes, not system RAM or Swap."""
    return {
        'active_bytes': mx.get_active_memory(),
        'cache_bytes': mx.get_cache_memory(),
        'peak_bytes': mx.get_peak_memory(),
    }


### 为什么求值边界必须放在观察之前？

[18.2](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/50aed835a74af0fe380b39b854329994908630c1/notebooks/78_step18_2_eval_boundary.ipynb)说明MLX可能先建立惰性计算依赖。若未来已有前节的一次真实训练更新，仅取得loss对象不代表新参数与Adam状态已经完成。用于观察该步结束状态的顺序是：

| 顺序 | 未来的操作 | 原因 |
|---|---|---|
| 1 | 按19.3规则检查并完成一次获准的稳定更新 | 有限值检查在optimizer update之前；不能用内存读数替代它 |
| 2 | `mx.eval(loss, model.state, optimizer.state)` | 请求完成该loss及当前模型/Adam数组 |
| 3 | 调用 `mlX_memory_snapshot_bytes()` | 读取这个明确边界之后的MLX计数 |
| 4 | 另列Activity Monitor观察及当时其他应用负载 | 系统层读数不能由步骤3推算 |

这里loss仍对应更新前参数，model/optimizer状态已更新；求值不会把loss改成更新后重新计算的loss。没有实际训练步时不能凭空填写这一观察表。本节没有执行上述任何一步。

求值完成也不等于释放仍被引用的数组。`mx.clear_cache()` 针对闲置MLX buffer，不删除存活的模型或Adam状态，也不是清空编译图缓存或Swap的命令。本节不调用它。固定来源：[memory.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/memory.cpp)。


## 5. Short bursts与sustained workload：一次结束说明不了持续负载

**Short burst（短时突发）** 是有界的短工作；**sustained workload（持续负载）** 会让处理器长时间工作。前者正常结束，最多说明那次调用结束，不证明后者不会出现压力积累或降频。温度与功耗还受到环境、散热设计、充电状态、其他进程和系统策略影响；本节没有采集其中任何指标。

未来观察先约定有界步骤及停止条件，而不是不断追加步骤直到看到理想数字。若系统持续高内存压力、Swap继续增长或交互明显失去响应，先停止追加工作，记录当时配置与其他负载，再讨论缩小候选或排查存活对象。不要在压力下反复清cache并把暂时变化当作问题已解决。

建议把记录分开：配置与shape；公式估算及dtype假设；MLX原始bytes和求值边界；系统Memory Pressure/Swap观察；短时还是持续负载。未观察项写“未测”，不能填0代替未知。没有温度读数时不作低温结论，单个快照也不证明整段负载无Swap。


## 6. 本节主线到此结束

现在应能区分：公式告诉你某张量的逻辑规模；候选函数只改配置；MLX计数描述其分配器；系统观察回答另一个层面的问题。它们合起来支持审慎选择，不能相互替代。

自查（不附答案）：

1. 给记录表标明哪些列来自公式、哪些必须由未来实际观察取得。
2. 为什么“当前active下降、cache上升”不能单独证明发生了内存泄漏或Swap增加？
3. 若只改block_size，应重新核对哪些数据窗口和模型容量约束？不要直接修改旧运行对象来代替说明。

下面是可选的旧依赖恢复区。只阅读本节重点时可以先停在这里。没有运行模型的要求，也没有已测得的资源结论。


## 附录A · 已学依赖：配置与数据函数（可选完整代码）

附录保留一套自包含依赖，按本Notebook顺序定义即可，不依赖旧kernel。来源是 [19.1配置](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/50aed835a74af0fe380b39b854329994908630c1/notebooks/82_step19_1_hyperparameter_baseline.ipynb)、[16.1数据来源](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/50aed835a74af0fe380b39b854329994908630c1/notebooks/67_step16_1_acquire_inspect_tiny_shakespeare.ipynb)、[16.2顺序划分](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/50aed835a74af0fe380b39b854329994908630c1/notebooks/68_step16_2_train_validation_split.ipynb)、[16.3采样](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/50aed835a74af0fe380b39b854329994908630c1/notebooks/69_step16_3_batch_sampler.ipynb) 和 [15.4完整模型](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/50aed835a74af0fe380b39b854329994908630c1/notebooks/65_step15_4_implement_minigpt.ipynb)。这里恢复已讲接口，不再引入新的训练方案。

正文已有math、mx和baseline_candidate；下面增加数据与模型定义所需导入、原seeds及配置映射。`make_optimizer_kwargs` 只保留原文件最后生效的一份，Adam仍显式使用bias_correction=True，与19.1–19.3一致；不是固定MLX版本的默认False。


In [ ]:
import random
from pathlib import Path
from urllib.request import urlopen
import mlx.nn as nn
import mlx.optimizers as optim


In [ ]:
seeds = {'model':2028,'train':2026,'validation':2027}

def make_model_kwargs(c,vocab_size):
    return {'vocab_size':vocab_size,'dims':c['model_width'],'num_heads':c['heads'],'num_blocks':c['layers'],'max_seq_len':c['block_size']}

def make_optimizer_kwargs(config):
    return {'learning_rate':config['learning_rate'],'betas':[0.9,0.999],'eps':1e-8,'bias_correction':True}


以下四个函数保持原接口：定位仓库、可选获取、读取真实缓存、90/10顺序划分。网络获取函数的存在不代表会调用；默认流程关闭获取，缓存缺失明确报错，不用toy文本代替真实语料。

读取函数将返回真实字符表、双向ID映射和一维整数stream；词表大小须从该字符表取得。下一段只是定义，尚无文件或网络访问。


In [ ]:
SOURCE_URL='https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'

def find_project_root(start):
    current=Path(start).resolve()
    for p in [current]+list(current.parents):
        if (p/'pyproject.toml').is_file() and (p/'notebooks'/'82_step19_1_hyperparameter_baseline.ipynb').is_file(): return p
    raise FileNotFoundError('Set PROJECT_ROOT to the repository root.')

def acquire_tiny_shakespeare(path,source_url=SOURCE_URL):
    path=Path(path)
    if path.is_file(): return path
    path.parent.mkdir(parents=True,exist_ok=True)
    with urlopen(source_url,timeout=30) as response: raw=response.read()
    tmp=path.with_suffix('.tmp'); tmp.write_bytes(raw); tmp.replace(path); return path


In [ ]:
def load_cached_character_corpus(path):
    path=Path(path)
    if not path.is_file(): raise FileNotFoundError(f'Cache missing: {path.resolve()}; correct path or explicitly acquire it.')
    text=path.read_text(encoding='utf-8'); chars=sorted(set(text)); stoi={ch:i for i,ch in enumerate(chars)}; itos={i:ch for i,ch in enumerate(chars)}
    return text,chars,stoi,itos,mx.array([stoi[ch] for ch in text],dtype=mx.int32)

def split_train_validation(stream,fraction=0.90):
    if len(stream.shape)!=1 or stream.shape[0]<2 or not 0<fraction<1: raise ValueError('Invalid stream/split')
    i=int(stream.shape[0]*fraction)
    if i<=0 or i>=stream.shape[0]: raise ValueError('Empty split')
    return stream[:i],stream[i:],i


### 主动读取入口：只有调用时才恢复数据

旧稿把数据读取放在顶层，按顺序运行会直接访问缓存；这里将同一流程收进 `prepare_cached_data`。调用者明确传入仓库路径和实际候选，默认 `acquire_missing_data=False`。函数固定使用 `data/tinyshakespeare/input.txt`，返回字典中的键对应原来的数据变量；没有秘密改写全局变量。

使用**传入候选**的block_size做两边T+1长度检查，避免改了候选却仍按旧T检查。返回前对两段token数组显式求值，这只发生在未来主动调用时。定义该函数不查找目录、不读取缓存、不下载、不采样。


In [ ]:
def prepare_cached_data(project_root, config, *, acquire_missing_data=False):
    root = find_project_root(project_root)
    cache_path = root / "data" / "tinyshakespeare" / "input.txt"
    if acquire_missing_data:
        acquire_tiny_shakespeare(cache_path)
    text, characters, stoi, itos, token_stream = load_cached_character_corpus(cache_path)
    train_tokens, val_tokens, split_index = split_train_validation(token_stream, 0.90)
    for split_name, split in (("train", train_tokens), ("validation", val_tokens)):
        if split.shape[0] < config["block_size"] + 1:
            raise ValueError(f"{split_name} split needs at least block_size + 1 tokens")
    mx.eval(train_tokens, val_tokens)
    return {
        "PROJECT_ROOT": root, "CACHE_PATH": cache_path,
        "text": text, "characters": characters, "stoi": stoi, "itos": itos,
        "token_stream": token_stream, "train_tokens": train_tokens,
        "val_tokens": val_tokens, "split_index": split_index,
        "vocab_size": len(characters),
    }


下面全部是注释，仅展示未来如何显式接回变量；不会因Run All而读取数据。字典键取值不是另一次读取。若以后主动执行调用，先确认路径与缓存；只有明确允许获取时才改变开关。资源概念的手算不需要进行这些操作。


In [ ]:
# 未来工作副本中的主动入口；目前仅注释，不取消注释、不执行。
# PROJECT_ROOT = Path("/absolute/path/to/mlx-minigpt-from-scratch")
# ACQUIRE_MISSING_DATA = False
# selected_config = baseline_candidate  # 或此前明确选择的新候选
# data = prepare_cached_data(PROJECT_ROOT, selected_config,
#                            acquire_missing_data=ACQUIRE_MISSING_DATA)
# CACHE_PATH = data["CACHE_PATH"]
# text, characters = data["text"], data["characters"]
# stoi, itos, token_stream = data["stoi"], data["itos"], data["token_stream"]
# train_tokens, val_tokens = data["train_tokens"], data["val_tokens"]
# split_index, vocab_size = data["split_index"], data["vocab_size"]


## 附录B · 已学sampler与完整模型：每个类只保留一份

[16.3](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/50aed835a74af0fe380b39b854329994908630c1/notebooks/69_step16_3_batch_sampler.ipynb)的get_batch接口不变：输入已选择的一维split、B、T和独立Python RNG，返回 `(X,Y,starts)`；X/Y均为(B,T)，Y已经右移一位。下面只定义函数，不调用采样，也不重新混合train/validation。


In [ ]:
def get_batch(data, batch_size: int, block_size: int, rng: random.Random):
    """Sample shifted (X, Y) from one already-selected integer token split."""
    if len(data.shape) != 1:
        raise ValueError('data must be one token stream with shape (N_split,)')
    if type(batch_size) is not int or batch_size < 1:
        raise ValueError('batch_size must be a positive Python int')
    if type(block_size) is not int or block_size < 1:
        raise ValueError('block_size must be a positive Python int')

    N_split = data.shape[0]
    if N_split < block_size + 1:
        raise ValueError('the selected split needs at least block_size + 1 tokens')

    starts = [
        rng.randrange(0, N_split - block_size)
        for _ in range(batch_size)
    ]

    xs = []
    ys = []
    for start in starts:
        chunk = data[start : start + block_size + 1]  # (T+1,)
        xs.append(chunk[:-1])                         # (T,)
        ys.append(chunk[1:])                          # (T,)

    X = mx.stack(xs, axis=0)  # (B,T)
    Y = mx.stack(ys, axis=0)  # (B,T)
    return X, Y, starts


### CausalMultiHeadAttention：原模型的注意力定义

以下是 [15.4完整模型](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/50aed835a74af0fe380b39b854329994908630c1/notebooks/65_step15_4_implement_minigpt.ipynb)已使用的类；细节来自 [13.3拆头](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/50aed835a74af0fe380b39b854329994908630c1/notebooks/54_step13_3_split_channels_into_heads.ipynb) 等前节。输入/输出都是(B,T,C)，内部scores为(B,H,T,T)，恰是正文估算的对象。这里只恢复一份完整定义，没有更换架构、bias、mask或归一化方式。

旧单元里同名底层类各出现12次、MiniGPT出现4次；逐一定义的静态结构相同，最后生效者也与15.4一致。保留下面一套足够，重复声明不增加模型层数，也不是教学所需的新实现。


In [ ]:
class CausalMultiHeadAttention(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()

        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        self.dims = dims
        self.num_heads = num_heads
        self.head_dim = dims // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)  # Python float, not trainable

        self.qkv_proj = nn.Linear(
            input_dims=dims,
            output_dims=3 * dims,
            bias=False,
        )
        self.c_proj = nn.Linear(
            input_dims=dims,
            output_dims=dims,
            bias=False,
        )

    def __call__(self, x: mx.array) -> mx.array:
        if len(x.shape) != 3:
            raise ValueError('attention input must have shape (B,T,C)')

        B, T, C = x.shape
        if C != self.dims:
            raise ValueError('attention input width must equal dims')

        H = self.num_heads
        D = self.head_dim

        qkv = self.qkv_proj(x)  # (B,T,3C)
        q, k, v = mx.split(qkv, indices_or_sections=3, axis=-1)

        q = mx.reshape(q, (B, T, H, D))
        k = mx.reshape(k, (B, T, H, D))
        v = mx.reshape(v, (B, T, H, D))

        q = mx.swapaxes(q, 1, 2)  # (B,H,T,D)
        k = mx.swapaxes(k, 1, 2)
        v = mx.swapaxes(v, 1, 2)

        scores = mx.matmul(
            q,
            mx.swapaxes(k, -1, -2),
        ) * self.scale  # (B,H,T,T)

        # IMPORTANT: structural mask is local to forward.
        # It is NOT stored as self.causal_mask.
        future = mx.triu(mx.ones((T, T)), k=1) > 0
        mask = mx.where(
            future,
            mx.array(float('-inf'), dtype=scores.dtype),
            mx.array(0.0, dtype=scores.dtype),
        )

        weights = mx.softmax(scores + mask, axis=-1)
        head_out = mx.matmul(weights, v)  # (B,H,T,D)

        token_major = mx.swapaxes(head_out, 1, 2)  # (B,T,H,D)
        merged = mx.reshape(token_major, (B, T, C))
        return self.c_proj(merged)


### FeedForwardMLP与TransformerBlock：复用已学组合

[14.4 MLP](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/50aed835a74af0fe380b39b854329994908630c1/notebooks/60_step14_4_feed_forward_mlp.ipynb)沿用带bias的C→4C→C与GELU；Block沿用两条Pre-Norm残差路径。它们保持(B,T,C)不变，真正的层数由MiniGPT中的num_blocks决定。以下定义与原文件最后生效的版本一致。


In [ ]:
class FeedForwardMLP(nn.Module):
    def __init__(self, dims: int):
        super().__init__()
        self.fc1 = nn.Linear(dims, 4 * dims, bias=True)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(4 * dims, dims, bias=True)

    def __call__(self, x: mx.array) -> mx.array:
        return self.fc2(self.act(self.fc1(x)))

class TransformerBlock(nn.Module):
    def __init__(self, dims: int, num_heads: int):
        super().__init__()
        self.norm1 = nn.RMSNorm(dims=dims)
        self.attn = CausalMultiHeadAttention(dims=dims, num_heads=num_heads)
        self.norm2 = nn.RMSNorm(dims=dims)
        self.mlp = FeedForwardMLP(dims=dims)

    def __call__(self, x: mx.array) -> mx.array:
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


### MiniGPT：可选的完整汇总定义

输入整数IDs为(B,T)，通过token/position embedding、N个Block和LM head，输出raw logits为(B,T,V)。`max_seq_len`限制位置表容量，`vocab_size`必须匹配实际映射。下方较长块是已学模型的完整复用，不是19.4新增知识；这里不会实例化它。


In [ ]:
class MiniGPT(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dims: int,
        num_heads: int,
        num_blocks: int,
        max_seq_len: int,
    ):
        super().__init__()

        if vocab_size <= 0:
            raise ValueError('vocab_size must be positive')
        if dims <= 0:
            raise ValueError('dims must be positive')
        if num_heads <= 0:
            raise ValueError('num_heads must be positive')
        if num_blocks <= 0:
            raise ValueError('num_blocks must be positive')
        if max_seq_len <= 0:
            raise ValueError('max_seq_len must be positive')
        if dims % num_heads != 0:
            raise ValueError('dims must be divisible by num_heads')

        # Python metadata: these are not trainable mx.array parameters.
        self.vocab_size = vocab_size
        self.dims = dims
        self.max_seq_len = max_seq_len

        # Learned lookup tables.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            dims=dims,
        )
        self.position_embedding = nn.Embedding(
            num_embeddings=max_seq_len,
            dims=dims,
        )

        # A list of distinct Module instances; MLX recursively tracks them.
        self.blocks = [
            TransformerBlock(
                dims=dims,
                num_heads=num_heads,
            )
            for _ in range(num_blocks)
        ]

        self.final_norm = nn.RMSNorm(dims=dims)
        self.lm_head = nn.Linear(
            input_dims=dims,
            output_dims=vocab_size,
            bias=False,
        )

    def __call__(self, token_ids: mx.array) -> mx.array:
        if len(token_ids.shape) != 2:
            raise ValueError('token_ids must have shape (B,T)')

        B, T = token_ids.shape

        if T < 1:
            raise ValueError('sequence length T must be at least 1')
        if T > self.max_seq_len:
            raise ValueError(
                f'sequence length {T} exceeds max_seq_len {self.max_seq_len}'
            )

        # Forward-local structural indices: not Module parameters.
        position_ids = mx.arange(T)  # (T,)

        token_features = self.token_embedding(token_ids)        # (B,T,C)
        position_features = self.position_embedding(position_ids)  # (T,C)
        x = token_features + position_features                  # (B,T,C)

        for block in self.blocks:
            x = block(x)  # invariant: (B,T,C)

        x = self.final_norm(x)  # (B,T,C)
        logits = self.lm_head(x)  # (B,T,V)

        expected_shape = (B, T, self.vocab_size)
        if logits.shape != expected_shape:
            raise RuntimeError(
                f'logits shape {logits.shape} != expected {expected_shape}'
            )

        return logits


## 附录C · 已学loss与未来初始化边界

[17.2 flatten监督](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/50aed835a74af0fe380b39b854329994908630c1/notebooks/73_step17_2_flatten_bt_supervision.ipynb)已将raw logits (B,T,V)与Y (B,T)配对展平，再对逐位置交叉熵取mean。get_batch已经shift过Y，这里不再shift。保留原loss_fn接口；定义函数不会forward或求梯度。


In [ ]:
def loss_fn(model,X,Y):
    logits=model(X); B,T,V=logits.shape
    flat_logits=mx.reshape(logits,(B*T,V)); flat_targets=mx.reshape(Y,(B*T,))
    losses=nn.losses.cross_entropy(flat_logits,flat_targets,axis=-1,reduction='none')
    return mx.mean(losses)


下面仍是**注释形式的未来setup**：只有主动执行时才初始化模型和Adam。必须使用与数据检查一致的selected_config和真实vocab_size；求导函数绑定的model与之后使用的model必须相同。training_state保存当前两个原状态容器的引用，不是冻结副本。

这只是恢复对象，不是训练更新。要真正训练，应回到 [19.3稳定性流程](https://github.com/kris0516/mlx-minigpt-from-scratch/blob/50aed835a74af0fe380b39b854329994908630c1/notebooks/84_step19_3_training_stability.ipynb) 的更新前有限值检查、范数/裁剪顺序，不能把下面的初始化当成完整stable_train_step。本节没有拼入训练/验证循环，也不会在读取、初始化后自动开跑。


In [ ]:
# 未来明确选择执行setup时才取消注释；本次均不执行。
# mx.random.seed(seeds["model"])
# model = MiniGPT(**make_model_kwargs(selected_config, vocab_size))
# loss_and_grad_fn = nn.value_and_grad(model, loss_fn)
# optimizer = optim.Adam(**make_optimizer_kwargs(selected_config))
# optimizer.init(model.trainable_parameters())
# training_state = [model.state, optimizer.state]
# train_rng = random.Random(seeds["train"])
# validation_rng = random.Random(seeds["validation"])


## 固定版本依据与完成边界

- [MLX v0.32.2 unified_memory.rst](https://github.com/ml-explore/mlx/blob/v0.32.2/docs/src/usage/unified_memory.rst)：CPU/GPU共享内存池。
- [v0.32.2 memory.cpp](https://github.com/ml-explore/mlx/blob/v0.32.2/python/src/memory.cpp)：三个计数getter的bytes单位、peak起点、cache含义及clear_cache接口。
- [v0.32.2 Metal allocator实现](https://github.com/ml-explore/mlx/blob/v0.32.2/mlx/backend/metal/allocator.cpp)：分配/回收与计数的实现边界。
- [Apple Activity Monitor内存说明](https://support.apple.com/guide/activity-monitor/view-memory-usage-actmntr1004/mac)：系统Memory Pressure与Swap等独立口径。

本次只改这一节的教学组织、去重和显式读取入口。原四个模型类、默认候选/种子及已有helper接口保持；数据恢复不再位于自动执行的顶层。没有运行Notebook、训练、验证、配置/测试代码、采样、测量、语料读取下载或用户工作副本，没有修改系统设置，也没有真实资源或性能结果。

19.4本次教学修订1/2；19.2和19.3各1/2等历史计数不变。所有代码单元execution_count=null、outputs=[]，材料发布和静态校验不等于运行验证或学员掌握。本轮在19.4结束，下一步由独立审阅决定。
